In [2]:
"""
merge_chunks.py

Combines the PDF-derived and Elsevier-derived chunk tables into one
unified corpus, ready for Phase 2 (triple extraction). Flags any DOI
that appears in both sources, e.g. a paper accidentally downloaded
both as a PDF and pulled again via the Elsevier TDM API, so you can
catch and drop the duplicate before running extraction twice on the
same paper.
"""

import pandas as pd


def run():
    df_pdf = pd.read_parquet("chunks_pdf.parquet")
    df_elsevier = pd.read_parquet("chunks_elsevier.parquet")

    combined = pd.concat([df_pdf, df_elsevier], ignore_index=True)

    # Detect DOIs present in both sources
    pdf_dois = set(df_pdf["doi"].unique())
    elsevier_dois = set(df_elsevier["doi"].unique())
    overlap = pdf_dois & elsevier_dois

    if overlap:
        print(f"WARNING: {len(overlap)} DOIs appear in both PDF and Elsevier sources:")
        for doi in sorted(overlap):
            print(f"  {doi}")
        print("Keeping Elsevier version for these, dropping PDF version.\n")
        # Drop PDF rows for any DOI that also has an Elsevier version
        combined = combined[
            ~((combined["source_type"] == "pdf") & (combined["doi"].isin(overlap)))
        ]

    combined = combined.reset_index(drop=True)
    combined.to_parquet("chunks_all.parquet", index=False)
    print(f"Merged corpus: {len(combined)} chunks from {combined['doi'].nunique()} unique papers")
    print(combined["source_type"].value_counts())


if __name__ == "__main__":
    run()

Merged corpus: 1108 chunks from 192 unique papers
source_type
elsevier_txt    745
pdf             363
Name: count, dtype: int64
